In [1]:
!pip install -U bitsandbytes>=0.46.1 transformers accelerate qwen-vl-utils turboquant

Decomposer Agent

In [2]:
# decomposer_base.py
from abc import ABC, abstractmethod
from typing import Dict, Any, Optional

class BaseClaimDecomposer(ABC):
    """
    Classe astratta per la scomposizione dei claim.
    """

    @abstractmethod
    def decompose(self, text_input: str) -> Optional[Dict[str, Any]]:
        """
        Riceve in input il testo e restituisce un JSON strutturato
        con i sub-claims e le strategie di routing.
        """
        pass

Prompt Builder

In [3]:
# prompt_utils.py

def build_qwen_few_shot_prompt(user_text: str, image_path: str = None) -> list:
    """
    Costruisce i messaggi in formato standard per Qwen2.5-VL.
    Restituisce una lista di dizionari pronta per essere processata da apply_chat_template().
    """

    # --- 1. SYSTEM PROMPT (Identità Strict e Regole per Classificazioni) ---
    system_prompt = """Sei uno script di estrazione dati MULTIMODALE e AUTOMATICO. Puoi analizzare sia testo che immagini. Il tuo unico scopo è estrarre un array di proposizioni dichiarative indipendenti (Soggetto + Verbo + Oggetto).
NON sei un assistente conversazionale. NON scusarti, NON dialogare e NON fare domande. Restituisci SEMPRE e SOLO il JSON valido. Se rifiuti di analizzare, lo script andrà in crash.

REGOLE SINTATTICHE (CRITICHE):
1. COMPLETEZZA: Estrai TUTTI i claim medici, chimici o scientifici rilevanti dal testo o dai contenuti dell'immagine.
2. ANEDDOTI E FAKE NEWS: Scarta rigorosamente le narrazioni personali ("Mio cugino ha preso..."). Tuttavia, se la frase nasconde una teoria medica (es. "X è la cura definitiva!"), ESTRAI QUELLA TEORIA in modo neutrale.
3. RISOLUZIONE DEL SOGGETTO: I pronomi o i soggetti sottintesi devono essere esplicitati.
4. REASONING BREVE: Il campo "reasoning" deve essere di MASSIMO 15 PAROLE per evitare errori di formattazione.

REGOLE DI CLASSIFICAZIONE (ROUTING) - LEGGERE ATTENTAMENTE:
- Assegna ["kb"] SOLO alle frasi che descrivono definizioni biologiche/chimiche, tassonomia, composizione o proprietà puramente statiche (es. "X è un ormone", "L'aspirina è un farmaco", "X contiene Y", "X ha una massa di Z").
- Assegna ["kb", "lit"] a TUTTE le altre frasi che descrivono cause, correlazioni, azioni, effetti clinici, cure o eventi (es. "X altera Y", "X riduce Z", "X è la causa di Y").

FORMATO DI OUTPUT:
{
  "reasoning": "Breve logica...",
  "sub_claims": [
    {
      "claim": "Frase completa e indipendente",
      "routes": ["..."]
    }
  ]
}"""

    # --- 2. LA STRUTTURA A MESSAGGI (Few-Shot Examples) ---
    messages = [
        # Inizializziamo il System Prompt
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},

        # ESEMPIO 1: Routing base
        {"role": "user", "content": [{"type": "text", "text": "TITOLO: La radice magica. PUNTO 1: Contiene molta vitamina C. PUNTO 2: Cura il cancro ai polmoni in una settimana."}]},
        {"role": "assistant", "content": [{"type": "text", "text": "{\n  \"reasoning\": \"Punto 1 composizione (kb). Punto 2 azione estrema (kb, lit). Soggetto risolto.\",\n  \"sub_claims\": [\n    {\n      \"claim\": \"La radice magica contiene molta vitamina C.\",\n      \"routes\": [\"kb\"]\n    },\n    {\n      \"claim\": \"La radice magica cura il cancro ai polmoni in una settimana.\",\n      \"routes\": [\"kb\", \"lit\"]\n    }\n  ]\n}"}]},

        # ESEMPIO 2: Eliminazione metodologie
        {"role": "user", "content": [{"type": "text", "text": "Uno studio dimostra che mangiare aglio altera i batteri intestinali. Ricordiamo che l'aglio è un bulbo."}]},
        {"role": "assistant", "content": [{"type": "text", "text": "{\n  \"reasoning\": \"Scarto la metodologia. Estraggo l'azione (kb, lit) e la definizione statica (kb).\",\n  \"sub_claims\": [\n    {\n      \"claim\": \"Mangiare aglio altera i batteri intestinali.\",\n      \"routes\": [\"kb\", \"lit\"]\n    },\n    {\n      \"claim\": \"L'aglio è un bulbo.\",\n      \"routes\": [\"kb\"]\n    }\n  ]\n}"}]},

        # ESEMPIO 3: Gestione severa di Aneddoti e Fake News mischiate (Questo risolve il Test 2)
        {"role": "user", "content": [{"type": "text", "text": "Mio nonno mangiava sempre la terra e il giorno dopo stava bene. I medici mentono, la terra è la cura definitiva! Ricordate che la terra contiene minerali."}]},
        {"role": "assistant", "content": [{"type": "text", "text": "{\n  \"reasoning\": \"Scarto aneddoto del nonno. Estraggo claim medico estremo (kb, lit) e composizione (kb).\",\n  \"sub_claims\": [\n    {\n      \"claim\": \"La terra è la cura definitiva.\",\n      \"routes\": [\"kb\", \"lit\"]\n    },\n    {\n      \"claim\": \"La terra contiene minerali.\",\n      \"routes\": [\"kb\"]\n    }\n  ]\n}"}]},

        # ESEMPIO 4: Routing per classificazioni farmaceutiche (Questo risolve il Test 6)
        {"role": "user", "content": [{"type": "text", "text": "L'ibuprofene è un antinfiammatorio non steroideo. Riduce drasticamente il dolore articolare."}]},
        {"role": "assistant", "content": [{"type": "text", "text": "{\n  \"reasoning\": \"Estraggo classificazione farmacologica (kb) e azione clinica (kb, lit).\",\n  \"sub_claims\": [\n    {\n      \"claim\": \"L'ibuprofene è un antinfiammatorio non steroideo.\",\n      \"routes\": [\"kb\"]\n    },\n    {\n      \"claim\": \"L'ibuprofene riduce drasticamente il dolore articolare.\",\n      \"routes\": [\"kb\", \"lit\"]\n    }\n  ]\n}"}]}
    ]

    # --- 3. GESTIONE DELL'INPUT REALE DELL'UTENTE (Con Multimodalità) ---
    user_content = []

    # Se c'è un'immagine, la aggiungiamo PRIMA del testo
    if image_path:
        user_content.append({"type": "image", "image": image_path})

    # Aggiungiamo il testo reale
    user_content.append({"type": "text", "text": user_text})

    # Aggiungiamo il messaggio dell'utente alla lista
    messages.append({"role": "user", "content": user_content})

    return messages

Decomposer Agent

In [4]:
# qwen_decomposer.py
import torch
import json
import requests
import gc
from io import BytesIO
from bs4 import BeautifulSoup
from PIL import Image
from urllib.parse import urlparse

# IMPORTAZIONE CORRETTA PER QWEN 2.5
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# from decomposer_base import BaseClaimDecomposer
# from prompt_utils import build_qwen_few_shot_prompt

class QwenNF4Decomposer(BaseClaimDecomposer):
    def __init__(self, model_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"):
        print(f"Caricamento di {model_id} in modalità Multimodale (NF4 + SDPA)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        self.processor = AutoProcessor.from_pretrained(model_id)

        # CARICAMENTO CON SDPA (Ottimizzazione nativa della memoria)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            attn_implementation="sdpa"
        )
        print("Modello Multimodale caricato e pronto!")

    def _is_url(self, text: str) -> bool:
        try:
            result = urlparse(text)
            return all([result.scheme, result.netloc])
        except ValueError:
            return False

    def _scrape_text_from_url(self, url: str) -> str:
        print(f"Scraping del contenuto da: {url}...")
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64 AppleWebKit/537.36)'}
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            for script in soup(["script", "style", "nav", "footer", "header"]):
                script.extract()

            text = soup.get_text(separator=' ', strip=True)
            return text[:4000] + ("..." if len(text) > 4000 else "")
        except Exception as e:
            print(f"Errore nello scraping URL: {e}")
            return f"Errore: Impossibile leggere {url}"

    def decompose(self, text_input: str, image_path: str = None) -> dict:
        if self._is_url(text_input):
            text_to_process = self._scrape_text_from_url(text_input)
            print("Testo estratto dall'URL con successo.")
        else:
            text_to_process = text_input

        # 1. Costruzione dei messaggi
        messages = build_qwen_few_shot_prompt(text_to_process, image_path)

        # 2. Template
        text_with_template = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # 3. Gestione Immagine (Sblocco server, fix PNG trasparenti e BytesIO)
        if image_path:
            print(f"Caricamento immagine da: {image_path}...")
            try:
                if self._is_url(image_path):
                    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64 AppleWebKit/537.36)'}
                    img_response = requests.get(image_path, headers=headers, timeout=15)
                    img_response.raise_for_status()
                    image = Image.open(BytesIO(img_response.content)).convert("RGB")
                else:
                    image = Image.open(image_path).convert("RGB")

                inputs = self.processor(
                    text=[text_with_template],
                    images=[image],
                    padding=True,
                    return_tensors="pt"
                ).to(self.model.device)
            except Exception as e:
                print(f"Errore caricamento immagine: {e}")
                inputs = self.processor(text=[text_with_template], return_tensors="pt").to(self.model.device)
        else:
            # IL BLOCCO CHE MANCAVA! Questo gestisce l'input solo testuale.
            inputs = self.processor(text=[text_with_template], return_tensors="pt").to(self.model.device)

        # 4. Generazione
        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True
            )

        # 5. Decodifica
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        generated_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        # 6. Parsing
        risultato_json = self._parse_json_output(generated_text)

        # 7. Pulizia Memoria VRAM
        del inputs
        del generated_ids
        del generated_ids_trimmed
        torch.cuda.empty_cache()
        gc.collect()

        return risultato_json

    def _parse_json_output(self, raw_text: str) -> dict:
        clean_text = raw_text.strip()
        if clean_text.startswith("```json"):
            clean_text = clean_text[7:]
        if clean_text.endswith("```"):
            clean_text = clean_text[:-3]

        clean_text = clean_text.replace(".\n  \"sub_claims\"", ".\",\n  \"sub_claims\"")

        try:
            parsed_json = json.loads(clean_text.strip())
            return parsed_json
        except json.JSONDecodeError as e:
            print(f"ERRORE DI PARSING JSON: {e}")
            print(f"Testo grezzo generato:\n{raw_text}")
            return {"sub_claims": []}

TEST

In [5]:
# main.py
#from qwen_decomposer import QwenNF4Decomposer

# Inizializziamo l'Agente Decomposer
decomposer_agent = QwenNF4Decomposer()

# Il claim in ingresso (ad es. estratto dalla dashboard)
user_tweet = "La carenza di vitamina D è la vera causa dell'aumento dei casi di asma nei bambini, inoltre la vitamina D è una vitamina liposolubile."

# Esecuzione
risultato = decomposer_agent.decompose(user_tweet)

print("\n--- RISULTATO FASE 1 (INGESTION & DECOMPOSITION) ---")
print(risultato)

Caricamento di Qwen/Qwen2.5-VL-7B-Instruct in modalità Multimodale (NF4 + SDPA)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Modello Multimodale caricato e pronto!

--- RISULTATO FASE 1 (INGESTION & DECOMPOSITION) ---
{'reasoning': 'Estraggo azione causale (kb, lit) e definizione chimica (kb).', 'sub_claims': [{'claim': "La carenza di vitamina D è la vera causa dell'aumento dei casi di asma nei bambini.", 'routes': ['kb', 'lit']}, {'claim': 'La vitamina D è una vitamina liposolubile.', 'routes': ['kb']}]}


Test Suite

In [6]:
import json

# Definiamo i casi di test
test_cases = [
    {
        "id": "T01",
        "type": "Basic Fact",
        "input": "Il gene BRCA1 è associato a un maggior rischio di sviluppare il tumore al seno.",
        "description": "Fatto medico semplice, ideale per la Knowledge Base."
    },
    {
        "id": "T02",
        "type": "Social Media Post (Tweet)",
        "input": "Mio cugino ha preso l'ivermectina e il giorno dopo non aveva più il Covid. I medici non ce lo dicono, ma è la cura definitiva! Ah, e ricordate che l'ivermectina è un antiparassitario per cavalli.",
        "description": "Linguaggio colloquiale, aneddotico, contiene un claim complesso e uno fattuale."
    },
    {
        "id": "T03",
        "type": "News Article Excerpt",
        "input": "Uno studio recente suggerisce che il digiuno intermittente prolungato (oltre le 16 ore) può alterare significativamente la flora intestinale e aumentare i livelli di cortisolo. Tuttavia, il cortisolo è un ormone steroideo essenziale per la vita.",
        "description": "Linguaggio giornalistico, richiede di scindere l'esito dello studio (lit) dalla definizione dell'ormone (kb)."
    },
    {
        "id": "T04",
        "type": "Simulated Image OCR (Infografica)",
        "input": "TITOLO: Benefici della curcuma. PUNTO 1: Contiene curcumina. PUNTO 2: Riduce l'infiammazione articolare meglio dell'ibuprofene. PUNTO 3: Cura il cancro al fegato in 30 giorni.",
        "description": "Testo schematico estratto da un'ipotetica infografica."
    },
    {
        "id": "T05",
        "type": "Complex URL Scrape (Abstract Medico)",
        "input": "BACKGROUND: L'efficacia degli inibitori SGLT2 nei pazienti con scompenso cardiaco a frazione di eiezione preservata (HFpEF) rimane dibattuta. METODI: Abbiamo condotto uno studio randomizzato in doppio cieco. RISULTATI: L'empagliflozin ha ridotto il rischio combinato di morte cardiovascolare o ospedalizzazione.",
        "description": "Linguaggio medico altamente specializzato."
    },
    {
        "id": "T06",
        "type": "Real URL Scraping",
        "input": "https://it.wikipedia.org/wiki/Acido_acetilsalicilico",
        "description": "Test dello scraper integrato su un URL reale."
    },
    {
        "id": "T07",
        "type": "Vision-Language (Image)",
        "input": "Analizza i claim e i dati chimici presenti in questa immagine.",
        # FIX: Usiamo l'API REST di PubChem che restituisce sempre un PNG pulito e sicuro! (CID 2244 = Aspirina)
        "image": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/2244/PNG",
        "description": "Test della capacità di vedere e interpretare immagini esterne."
    }
]

# Funzione di esecuzione dei test
def run_test_suite(decomposer_agent):
    print("🚀 INIZIO ESECUZIONE TEST SUITE DECOMPOSER 🚀\n")
    print("-" * 50)

    passed_tests = 0
    total_tests = len(test_cases)

    for i, test in enumerate(test_cases):
        print(f"▶️ TEST {i+1}: {test['id']} - {test['type']}")
        print(f"Descrizione: {test['description']}")
        print(f"Input: \"{test['input']}\"\n")

        try:
            # FIX: Gestione corretta dell'immagine nel dizionario
            if "image" in test:
                result = decomposer_agent.decompose(test['input'], image_path=test['image'])
            else:
                result = decomposer_agent.decompose(test['input'])

            print("Output Generato:")
            print(json.dumps(result, indent=2, ensure_ascii=False))

            # Verifica base: ha restituito una lista di sub_claims?
            if result and "sub_claims" in result and len(result["sub_claims"]) > 0:
                print("\n✅ STATO: Test Completato con Successo")
                passed_tests += 1
            else:
                print("\n❌ STATO: Fallito (Nessun sub_claim valido estratto)")

        except Exception as e:
            print(f"\n❌ STATO: Errore di Esecuzione - {str(e)}")

        print("-" * 50)

    print(f"\n📊 RISULTATI FINALI: {passed_tests}/{total_tests} Test Passati")

# Esecuzione
# Assicurati di aver inizializzato il modello: decomposer_agent = QwenNF4Decomposer()
run_test_suite(decomposer_agent)

🚀 INIZIO ESECUZIONE TEST SUITE DECOMPOSER 🚀

--------------------------------------------------
▶️ TEST 1: T01 - Basic Fact
Descrizione: Fatto medico semplice, ideale per la Knowledge Base.
Input: "Il gene BRCA1 è associato a un maggior rischio di sviluppare il tumore al seno."

Output Generato:
{
  "reasoning": "Estraggo relazione genetica (kb, lit).",
  "sub_claims": [
    {
      "claim": "Il gene BRCA1 è associato a un maggior rischio di sviluppare il tumore al seno.",
      "routes": [
        "kb",
        "lit"
      ]
    }
  ]
}

✅ STATO: Test Completato con Successo
--------------------------------------------------
▶️ TEST 2: T02 - Social Media Post (Tweet)
Descrizione: Linguaggio colloquiale, aneddotico, contiene un claim complesso e uno fattuale.
Input: "Mio cugino ha preso l'ivermectina e il giorno dopo non aveva più il Covid. I medici non ce lo dicono, ma è la cura definitiva! Ah, e ricordate che l'ivermectina è un antiparassitario per cavalli."

Output Generato:
{
  "re